In [1]:
import pandas as pd

df = pd.read_csv("data/Keylogger_Detection.csv")
print(df.shape)
df.head()

(523617, 86)


/var/folders/j_/x0rzqtss1wgdv8tmm18x55mw0000gn/T/ipykernel_1657/3643055495.py:3: DtypeWarning: Columns (48,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/Keylogger_Detection.csv")


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Class
0,0,10.42.0.211-52.6.25.230-34451-443-6,10.42.0.211,34451.0,52.6.25.230,443.0,6.0,04/08/2017 05:12:36,12140931.0,9.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,1,172.217.3.99-10.42.0.151-443-53892-6,10.42.0.151,53892.0,172.217.3.99,443.0,6.0,04/08/2017 07:55:51,418882.0,102.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,2,172.217.3.98-10.42.0.151-443-50750-6,172.217.3.98,443.0,10.42.0.151,50750.0,6.0,04/08/2017 08:48:19,45.0,2.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,3,10.42.0.211-10.42.0.1-23025-53-17,10.42.0.211,23025.0,10.42.0.1,53.0,17.0,04/08/2017 05:54:10,541699.0,1.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,4,10.42.0.211-123.129.244.226-52602-443-6,10.42.0.211,52602.0,123.129.244.226,443.0,6.0,04/08/2017 08:44:25,7310795.0,3.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [2]:
df = pd.read_csv("data/Keylogger_Detection.csv", low_memory=False)
print(df.columns[-5:].tolist())
print(df.columns[[48, 56]].tolist())
print(df.iloc[:, -1].value_counts())

['Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', 'Class']
[' Packet Length Std', ' CWE Flag Count']
Class
Benign       308813
Keylogger    214804
Name: count, dtype: int64


In [3]:
df.columns = df.columns.str.strip()

print("Valores em falta:", df.isna().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())
print(df.dtypes.value_counts())

for c in ["Packet Length Std", "CWE Flag Count"]:
    invalidos = pd.to_numeric(df[c], errors="coerce").isna() & df[c].notna()
    print(c, "-> valores não numéricos:", invalidos.sum(), df.loc[invalidos, c].unique()[:5])

Valores em falta: 918
Linhas duplicadas: 0
float64    78
object      7
int64       1
Name: count, dtype: int64
Packet Length Std -> valores não numéricos: 2 ['SCAREWARE']
CWE Flag Count -> valores não numéricos: 3 ['SCAREWARE']


In [4]:
mask = (df["Packet Length Std"] == "SCAREWARE") | (df["CWE Flag Count"] == "SCAREWARE")
print(df.loc[mask, ["Flow ID", "Source IP", "Packet Length Std", "CWE Flag Count", "Class"]])

faltas = df.isna().sum()
print(faltas[faltas > 0])


       Flow ID Source IP Packet Length Std CWE Flag Count      Class
59378      NaN         0         SCAREWARE            NaN  Keylogger
121271     NaN     281.0               0.0      SCAREWARE  Keylogger
463309     NaN         0         SCAREWARE            NaN  Keylogger
488826     NaN     281.0               0.0      SCAREWARE  Keylogger
499706     NaN     281.0               0.0      SCAREWARE  Keylogger
Flow ID           7
Flow IAT Mean     3
Flow IAT Std      3
Flow IAT Max      3
Flow IAT Min      3
                 ..
Active Min       22
Idle Mean        22
Idle Std         22
Idle Max         22
Idle Min         22
Length: 63, dtype: int64


In [5]:
import numpy as np

linhas_com_nan = df.isna().any(axis=1)
print("Linhas com pelo menos 1 valor em falta:", linhas_com_nan.sum())
print(df.loc[linhas_com_nan, "Class"].value_counts())

num = df.select_dtypes(include="number")
print("Valores infinitos:", np.isinf(num).sum().sum())

Linhas com pelo menos 1 valor em falta: 24
Class
Keylogger    24
Name: count, dtype: int64
Valores infinitos: 0


In [6]:
antes = len(df)

# 1. "SCAREWARE" passa a NaN
for c in ["Packet Length Std", "CWE Flag Count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 2. Remover linhas com valores em falta
df = df.dropna()

# 3. Remover colunas de identificação
df = df.drop(columns=["Unnamed: 0", "Flow ID", "Source IP", "Destination IP", "Timestamp"])

print("Linhas removidas:", antes - len(df))
print("Shape:", df.shape)
print("Duplicadas:", df.duplicated().sum())
print(df.dtypes.value_counts())
print(df["Class"].value_counts())

Linhas removidas: 24
Shape: (523593, 81)
Duplicadas: 357714
float64    80
object      1
Name: count, dtype: int64
Class
Benign       308813
Keylogger    214780
Name: count, dtype: int64


In [7]:
feats = [c for c in df.columns if c != "Class"]

sem_dup = df.drop_duplicates()
sem_dup_feats = df.drop_duplicates(subset=feats)

print("Únicas (features + Class):", len(sem_dup))
print("Únicas (só features):", len(sem_dup_feats))
print(sem_dup["Class"].value_counts())

Únicas (features + Class): 165879
Únicas (só features): 165879
Class
Benign       97856
Keylogger    68023
Name: count, dtype: int64


In [8]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape final:", df.shape)

df.to_csv("data/keylogger_clean.csv", index=False)


Shape final: (165879, 81)


In [9]:
from src.data_prep import load_raw, clean_dataset

df_teste = clean_dataset(load_raw("data/Keylogger_Detection.csv"))
print(df_teste.shape)

(165879, 81)


In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="Class")
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Treino: (132703, 80)
Teste: (33176, 80)
Class
Benign       0.589926
Keylogger    0.410074
Name: proportion, dtype: float64
Class
Benign       0.589914
Keylogger    0.410086
Name: proportion, dtype: float64


In [11]:
from src.data_prep import load_raw, clean_dataset

df = clean_dataset(load_raw("data/Keylogger_Detection.csv"))
print(df.shape)

(165879, 81)


In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="Class")
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Treino: (132703, 80)
Teste: (33176, 80)
Class
Benign       0.589926
Keylogger    0.410074
Name: proportion, dtype: float64
Class
Benign       0.589914
Keylogger    0.410086
Name: proportion, dtype: float64


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/p

              precision    recall  f1-score   support

      Benign       0.61      0.92      0.73     19571
   Keylogger       0.56      0.14      0.22     13605

    accuracy                           0.60     33176
   macro avg       0.59      0.53      0.48     33176
weighted avg       0.59      0.60      0.52     33176

[[18103  1468]
 [11710  1895]]


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [14]:
resumo = X_train.describe().T[["min", "max", "mean", "std"]]
resumo["amplitude"] = resumo["max"] - resumo["min"]
print(resumo.sort_values("max", ascending=False).head(10))

                               min           max          mean           std  \
Bwd Header Length    -1.353210e+11  1.551777e+10 -1.139147e+06  3.875431e+08   
Fwd Header Length    -7.533332e+10  1.040202e+10 -6.232530e+05  2.165497e+08   
Fwd Header Length.1  -7.533332e+10  1.040202e+10 -6.232530e+05  2.165497e+08   
min_seg_size_forward -1.395062e+09  1.705248e+08 -2.474022e+04  5.736308e+06   
Flow Duration         1.000000e+00  1.199967e+08  1.111790e+07  2.298562e+07   
Fwd IAT Total         0.000000e+00  1.199967e+08  8.070020e+06  2.007574e+07   
Bwd IAT Total         0.000000e+00  1.199487e+08  5.640198e+06  1.831832e+07   
Flow IAT Mean         1.000000e+00  1.199117e+08  2.842005e+06  8.057738e+06   
Fwd IAT Min           0.000000e+00  1.199117e+08  1.794134e+06  7.768560e+06   
Fwd IAT Max           0.000000e+00  1.199117e+08  6.717582e+06  1.624673e+07   

                         amplitude  
Bwd Header Length     1.508387e+11  
Fwd Header Length     8.573534e+10  
Fwd Head

In [15]:
num_cols = X_train.select_dtypes(include="number").columns

limites = X_train[num_cols].quantile([0.01, 0.99])

X_train_capped = X_train.copy()
X_test_capped = X_test.copy()

for col in num_cols:
    lo, hi = limites.loc[0.01, col], limites.loc[0.99, col]
    X_train_capped[col] = X_train_capped[col].clip(lo, hi)
    X_test_capped[col] = X_test_capped[col].clip(lo, hi)

print(X_train_capped[["Bwd Header Length", "Fwd Header Length"]].describe())

       Bwd Header Length  Fwd Header Length
count      132703.000000      132703.000000
mean          202.706194         187.528730
std           604.557085         400.079265
min             0.000000          20.000000
25%             0.000000          32.000000
50%            32.000000          64.000000
75%           132.000000         160.000000
max          4584.000000        2932.000000


In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_capped)
X_test_scaled = scaler.transform(X_test_capped)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/p

              precision    recall  f1-score   support

      Benign       0.61      0.92      0.73     19571
   Keylogger       0.56      0.15      0.23     13605

    accuracy                           0.60     33176
   macro avg       0.59      0.53      0.48     33176
weighted avg       0.59      0.60      0.53     33176

[[18020  1551]
 [11598  2007]]


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [17]:
# 1. Colunas duplicadas (mesmo conteúdo, nomes diferentes)
duplicadas = X_train.T[X_train.T.duplicated()].index.tolist()
print("Colunas com conteúdo duplicado:", duplicadas)

# 2. Colunas constantes (variação zero)
constantes = X_train.columns[X_train.nunique() <= 1].tolist()
print("Colunas constantes:", constantes)

Colunas com conteúdo duplicado: ['Fwd URG Flags', 'Bwd URG Flags', 'SYN Flag Count', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes']
Colunas constantes: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [18]:
colunas_a_remover = sorted(set(duplicadas) | set(constantes))
print(f"Total de colunas a remover: {len(colunas_a_remover)}")
print(colunas_a_remover)

X_train_capped = X_train_capped.drop(columns=colunas_a_remover)
X_test_capped = X_test_capped.drop(columns=colunas_a_remover)

print("Shape treino:", X_train_capped.shape)

Total de colunas a remover: 18
['Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Header Length.1', 'Fwd URG Flags', 'RST Flag Count', 'SYN Flag Count', 'Subflow Bwd Bytes', 'Subflow Bwd Packets', 'Subflow Fwd Bytes', 'Subflow Fwd Packets']
Shape treino: (132703, 62)


In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_capped)
X_test_scaled = scaler.transform(X_test_capped)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/p

              precision    recall  f1-score   support

      Benign       0.61      0.92      0.73     19571
   Keylogger       0.56      0.15      0.23     13605

    accuracy                           0.60     33176
   macro avg       0.59      0.53      0.48     33176
weighted avg       0.59      0.60      0.53     33176

[[18023  1548]
 [11603  2002]]


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [20]:
modelo = LogisticRegression(max_iter=2000, C=0.01, random_state=42)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/p

              precision    recall  f1-score   support

      Benign       0.61      0.93      0.73     19571
   Keylogger       0.57      0.13      0.22     13605

    accuracy                           0.60     33176
   macro avg       0.59      0.53      0.48     33176
weighted avg       0.59      0.60      0.52     33176

[[18178  1393]
 [11769  1836]]


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_capped, y_train)

y_pred_rf = rf.predict(X_test_capped)

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

      Benign       0.75      0.84      0.80     19571
   Keylogger       0.73      0.61      0.66     13605

    accuracy                           0.75     33176
   macro avg       0.74      0.72      0.73     33176
weighted avg       0.74      0.75      0.74     33176

[[16458  3113]
 [ 5344  8261]]


In [22]:
import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/random_forest_keylogger.joblib")
joblib.dump(scaler, "models/scaler_keylogger.joblib")
joblib.dump(modelo, "models/logistic_regression_keylogger.joblib")

['models/logistic_regression_keylogger.joblib']

In [23]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, max_depth=10)
dt.fit(X_train_capped, y_train)

y_pred_dt = dt.predict(X_test_capped)

print(classification_report(y_test, y_pred_dt))
print(confusion_matrix(y_test, y_pred_dt))

              precision    recall  f1-score   support

      Benign       0.65      0.87      0.75     19571
   Keylogger       0.64      0.34      0.45     13605

    accuracy                           0.65     33176
   macro avg       0.65      0.60      0.60     33176
weighted avg       0.65      0.65      0.62     33176

[[16943  2628]
 [ 8925  4680]]


In [24]:
joblib.dump(dt, "models/decision_tree_keylogger.joblib")

['models/decision_tree_keylogger.joblib']

In [25]:
importancias = pd.Series(rf.feature_importances_, index=X_train_capped.columns)
importancias = importancias.sort_values(ascending=False)

print("Top 15 features mais importantes:")
print(importancias.head(15))

print("\nBottom 10 features menos importantes:")
print(importancias.tail(10))

Top 15 features mais importantes:
Source Port                0.070401
Flow IAT Min               0.052579
Flow IAT Max               0.052169
Flow Duration              0.050780
Fwd Packets/s              0.049170
Flow Packets/s             0.048764
Flow IAT Mean              0.048111
Init_Win_bytes_forward     0.047667
Bwd Packets/s              0.033275
Fwd IAT Min                0.031488
Fwd IAT Max                0.030406
Fwd IAT Total              0.029537
Init_Win_bytes_backward    0.029157
Fwd IAT Mean               0.028302
Flow Bytes/s               0.025209
dtype: float64

Bottom 10 features menos importantes:
Active Mean         0.003383
act_data_pkt_fwd    0.003234
Down/Up Ratio       0.002044
Idle Std            0.001052
ACK Flag Count      0.001018
Fwd PSH Flags       0.000719
PSH Flag Count      0.000474
Active Std          0.000453
FIN Flag Count      0.000305
Protocol            0.000238
dtype: float64


In [26]:
print(X_train_capped.groupby(y_train)["Source Port"].describe())

             count          mean           std   min       25%      50%  \
Class                                                                     
Benign     78285.0  38241.419340  18852.747411  80.0  33928.00  43144.0   
Keylogger  54418.0  38672.321769  18458.726652  80.0  34169.25  43445.5   

               75%      max  
Class                        
Benign     52316.0  61773.0  
Keylogger  52332.0  61773.0  


In [27]:
baixa_importancia = importancias[importancias < 0.005].index.tolist()
print(f"Colunas a remover: {len(baixa_importancia)}")
print(baixa_importancia)

X_train_fs = X_train_capped.drop(columns=baixa_importancia)
X_test_fs = X_test_capped.drop(columns=baixa_importancia)

rf_fs = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_fs.fit(X_train_fs, y_train)

y_pred_fs = rf_fs.predict(X_test_fs)

print(classification_report(y_test, y_pred_fs))
print(confusion_matrix(y_test, y_pred_fs))

Colunas a remover: 19
['Bwd Packet Length Min', 'Total Backward Packets', 'Min Packet Length', 'Fwd Packet Length Min', 'Idle Max', 'Idle Min', 'Idle Mean', 'Active Min', 'Active Max', 'Active Mean', 'act_data_pkt_fwd', 'Down/Up Ratio', 'Idle Std', 'ACK Flag Count', 'Fwd PSH Flags', 'PSH Flag Count', 'Active Std', 'FIN Flag Count', 'Protocol']
              precision    recall  f1-score   support

      Benign       0.75      0.84      0.79     19571
   Keylogger       0.73      0.60      0.66     13605

    accuracy                           0.74     33176
   macro avg       0.74      0.72      0.73     33176
weighted avg       0.74      0.74      0.74     33176

[[16459  3112]
 [ 5386  8219]]


In [28]:
joblib.dump(rf_fs, "models/random_forest_keylogger_final.joblib")

import json
with open("models/features_keylogger.json", "w") as f:
    json.dump(list(X_train_fs.columns), f)

print("Guardado.")

Guardado.


In [1]:
def adicionar_features(X):
    X = X.copy()
    X["Fwd_Bwd_Byte_Ratio"] = X["Total Length of Fwd Packets"] / (X["Total Length of Bwd Packets"] + 1)
    return X

X_train_fe = adicionar_features(X_train_fs)
X_test_fe = adicionar_features(X_test_fs)

rf_fe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_fe.fit(X_train_fe, y_train)

y_pred_fe = rf_fe.predict(X_test_fe)

print(classification_report(y_test, y_pred_fe))

NameError: name 'X_train_fs' is not defined

In [2]:
from src.data_prep import load_raw, clean_dataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib, json

# 1. Limpeza
df = clean_dataset(load_raw("data/Keylogger_Detection.csv"))

# 2. Split
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Outlier capping
num_cols = X_train.select_dtypes(include="number").columns
limites = X_train[num_cols].quantile([0.01, 0.99])
X_train_capped = X_train.copy()
X_test_capped = X_test.copy()
for col in num_cols:
    lo, hi = limites.loc[0.01, col], limites.loc[0.99, col]
    X_train_capped[col] = X_train_capped[col].clip(lo, hi)
    X_test_capped[col] = X_test_capped[col].clip(lo, hi)

# 4. Remover colunas duplicadas/constantes
duplicadas = X_train.T[X_train.T.duplicated()].index.tolist()
constantes = X_train.columns[X_train.nunique() <= 1].tolist()
colunas_a_remover = sorted(set(duplicadas) | set(constantes))
X_train_capped = X_train_capped.drop(columns=colunas_a_remover)
X_test_capped = X_test_capped.drop(columns=colunas_a_remover)

# 5. Feature selection (usando a lista já guardada)
with open("models/features_keylogger.json") as f:
    features_finais = json.load(f)
X_train_fs = X_train_capped[features_finais]
X_test_fs = X_test_capped[features_finais]

print("Tudo recarregado. Shape:", X_train_fs.shape)

Tudo recarregado. Shape: (132703, 43)


In [3]:
def adicionar_features(X):
    X = X.copy()
    X["Fwd_Bwd_Byte_Ratio"] = X["Total Length of Fwd Packets"] / (X["Total Length of Bwd Packets"] + 1)
    return X

X_train_fe = adicionar_features(X_train_fs)
X_test_fe = adicionar_features(X_test_fs)

rf_fe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_fe.fit(X_train_fe, y_train)

y_pred_fe = rf_fe.predict(X_test_fe)

print(classification_report(y_test, y_pred_fe))

              precision    recall  f1-score   support

      Benign       0.75      0.84      0.79     19571
   Keylogger       0.72      0.60      0.66     13605

    accuracy                           0.74     33176
   macro avg       0.74      0.72      0.73     33176
weighted avg       0.74      0.74      0.74     33176

